## CREDIT PREDICTION

Load Data

In [ ]:
from snowflake.snowpark.context import get_active_session
import warnings

warnings.filterwarnings('ignore')

session = get_active_session()
session.sql("USE DATABASE ACCEPTION").collect()
session.sql("USE SCHEMA CREDIT_SCORE").collect()

train_df = session.read.option("SKIP_HEADER", 1).option("FIELD_OPTIONALLY_ENCLOSED_BY", '"').csv("@STG_DATA_CREDIT_SCORE/train.csv")
test_df = session.read.option("SKIP_HEADER", 1).option("FIELD_OPTIONALLY_ENCLOSED_BY", '"').csv("@STG_DATA_CREDIT_SCORE/test.csv")

column_names = [
    "ID", "CUSTOMER_ID", "MONTH", "NAME", "AGE", "SSN", "OCCUPATION", "ANNUAL_INCOME",
    "MONTHLY_INHAND_SALARY", "NUM_BANK_ACCOUNTS", "NUM_CREDIT_CARD", "INTEREST_RATE",
    "NUM_OF_LOAN", "TYPE_OF_LOAN", "DELAY_FROM_DUE_DATE", "NUM_OF_DELAYED_PAYMENT",
    "CHANGED_CREDIT_LIMIT", "NUM_CREDIT_INQUIRIES", "CREDIT_MIX", "OUTSTANDING_DEBT",
    "CREDIT_UTILIZATION_RATIO", "CREDIT_HISTORY_AGE", "PAYMENT_OF_MIN_AMOUNT",
    "TOTAL_EMI_PER_MONTH", "AMOUNT_INVESTED_MONTHLY", "PAYMENT_BEHAVIOUR",
    "MONTHLY_BALANCE", "CREDIT_SCORE"
]

old_cols_train = train_df.columns
for old, new in zip(old_cols_train, column_names):
    train_df = train_df.with_column_renamed(old, new)

old_cols_test = test_df.columns
test_column_names = column_names[:-1]
for old, new in zip(old_cols_test, test_column_names):
    test_df = test_df.with_column_renamed(old, new)

train_df = train_df.distinct()
test_df = test_df.distinct()

print(f"Train shape: {train_df.count()} rows, {len(train_df.columns)} columns")
print(f"Test shape: {test_df.count()} rows, {len(test_df.columns)} columns")
print(f"\nColunas train: {train_df.columns}")
print(f"Colunas test: {test_df.columns}")

Baseline Model

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

train_pd = train_df.to_pandas()
print(f"Colunas: {train_pd.columns.tolist()}")

label_col = train_pd.columns[-1]
print(f"Coluna target: {label_col}")

def BaseLine(df, target_col):
    df = df.dropna()
    features = df.drop([target_col], axis=1)
    target = df[target_col]
    
    cat_features = features.select_dtypes('object')
    for col in cat_features.columns:
        features[col], _ = pd.factorize(features[col])
    
    model = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.7, C=1, multi_class='multinomial', max_iter=200)
    Pipe_lr = Pipeline([('scaler', StandardScaler()), ('lr_model', model)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(estimator=Pipe_lr, X=features, y=target, cv=cv,
                            scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr'])
    report = pd.DataFrame(scores)
    return report

base = BaseLine(train_pd, label_col)
base

In [ ]:
# Ver estrutura e colunas do dataset
print("Colunas:", train_df.columns)
train_df.limit(5).to_pandas()

Exploratory Data (EDA)

In [ ]:
train_pd.info()

In [ ]:
import matplotlib.pyplot as plt

X = train_pd.drop(['CREDIT_SCORE', 'ID', 'CUSTOMER_ID'], axis=1)
y = train_pd['CREDIT_SCORE']

ForPie = []
for col, val in X.nunique().items():
    if 2 < val <= 16:
        ForPie.append(col)

fig, axes = plt.subplots(1, len(ForPie), figsize=(9*len(ForPie), 9))
if len(ForPie) == 1:
    axes = [axes]

for ax, col in zip(axes, ForPie):
    counts = X[col].value_counts()
    ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%', textprops={'fontsize': 12})
    ax.set_title(col, fontsize=14)

plt.suptitle('Percentage of labels of each cat. column', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
cat_features = X.select_dtypes('object')
num_features = X.select_dtypes(include= ['float', 'int'])

# null values..
print('Persentages of null values of each column:')
((train_pd.isnull().sum().sort_values(ascending=False)*100)/train_pd.shape[0]).head(10)

In [ ]:
cat_features.head()

In [ ]:
cols_to_drop = ['SSN', 'NAME', 'MONTH']
X = X.drop([c for c in cols_to_drop if c in X.columns], axis=1)

test_pd = test_df.to_pandas()
cols_to_drop_test = ['ID', 'CUSTOMER_ID', 'SSN', 'NAME', 'MONTH']
X_test = test_pd.drop([c for c in cols_to_drop_test if c in test_pd.columns], axis=1)

print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")

Feature Engineering

In [ ]:
def no_loans(df):
    if 'TYPE_OF_LOAN' in df.columns:
        df['NO_OF_LOAN'] = df['TYPE_OF_LOAN'].str.count(',') + 1
        cols_to_drop = [c for c in ['TYPE_OF_LOAN', 'NUM_OF_LOAN'] if c in df.columns]
        df.drop(cols_to_drop, axis=1, inplace=True)

no_loans(X)
no_loans(X_test)
print(f"X shape: {X.shape}, X_test shape: {X_test.shape}")

In [ ]:
X['PAYMENT_BEHAVIOUR']. unique()

In [ ]:
import numpy as np

X['PAYMENT_BEHAVIOUR'] = X['PAYMENT_BEHAVIOUR'].replace(['!@9#%8'], np.nan)
X_test['PAYMENT_BEHAVIOUR'] = X_test['PAYMENT_BEHAVIOUR'].replace(['!@9#%8'], np.nan)

def PB_handling(df):
    df[['SPEND', 'VALUE_PAYMENTS']] = df['PAYMENT_BEHAVIOUR'].str.split('_', expand=True).iloc[:, [0,2]]
    df.drop(['PAYMENT_BEHAVIOUR'], axis=1, inplace=True)

PB_handling(X)
PB_handling(X_test)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def Count_Plot(df, title=''):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for i, col in enumerate(['SPEND', 'VALUE_PAYMENTS']):
        sns.countplot(data=df, x=col, ax=ax[i], palette='deep')
        ax[i].set_title(f'Labels count of {col}')

    fig.suptitle(title, fontsize=13, color='darkblue')
    plt.tight_layout()
    plt.show()


Count_Plot(X, 'Counts of Train data Selected Features')

In [ ]:
cols = ['Years', 'Months']
to_break = 'CREDIT_HISTORY_AGE'

X = pd.concat([X, y], axis=1)

def CH_handling(df, cols, to_break):
    if to_break in df.columns:
        split_cols = df[to_break].str.split(" ", expand=True)
        if split_cols.shape[1] >= 4:
            df[cols] = split_cols.iloc[:, [0, 3]]
        df.dropna(subset=cols, inplace=True)
        df.drop([to_break], axis=1, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

X = CH_handling(X, cols, to_break)
X_test = CH_handling(X_test, cols, to_break)

print(f"X shape: {X.shape}, X_test shape: {X_test.shape}")

In [ ]:
y = X['CREDIT_SCORE']
X = X.drop(['CREDIT_SCORE'], axis=1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

to_float = ['ANNUAL_INCOME', 'CHANGED_CREDIT_LIMIT', 'OUTSTANDING_DEBT', 'AMOUNT_INVESTED_MONTHLY', 'MONTHLY_BALANCE']
to_int = ['AGE', 'NUM_OF_DELAYED_PAYMENT', 'Years', 'Months']

def converter(df, fcols, icols):
    fcols_exist = [c for c in fcols if c in df.columns]
    icols_exist = [c for c in icols if c in df.columns]
    
    if fcols_exist:
        df[fcols_exist] = df[fcols_exist].apply(pd.to_numeric, errors='coerce')
        df[fcols_exist] = df[fcols_exist].astype('float64')
    
    if icols_exist:
        df[icols_exist] = df[icols_exist].apply(pd.to_numeric, errors='coerce')
        df[icols_exist] = df[icols_exist].astype('Int64')

    if 'Years' in df.columns and 'Months' in df.columns:
        df['CREDIT_HISTORY_MONTHS'] = df['Years']*12 + df['Months']
        df['CREDIT_HISTORY_MONTHS'] = pd.to_numeric(df['CREDIT_HISTORY_MONTHS'], errors='coerce')
        df['CREDIT_HISTORY_MONTHS'] = df['CREDIT_HISTORY_MONTHS'].astype('Int64')
        df.drop(['Years', 'Months'], axis=1, inplace=True)

converter(X, to_float, to_int)
converter(X_test, to_float, to_int)

to_int_plot = ['AGE', 'NUM_OF_DELAYED_PAYMENT', 'CREDIT_HISTORY_MONTHS']

def histo(df, cols):
    cols_exist = [c for c in cols if c in df.columns]
    r, c = 1, len(cols_exist)
    fig, ax = plt.subplots(r, c, figsize=(17, 5))
    if len(cols_exist) == 1:
        ax = [ax]
    for i, col in enumerate(cols_exist):
        sns.histplot(data=df, x=col, color='purple', bins=30, ax=ax[i], kde=True)
        ax[i].set_title(f"Dist. of {col}\nskew = {df[col].skew():.2f}")
    plt.tight_layout()
    plt.show()

histo(X, to_float)

In [ ]:
histo(X, to_int)

In [ ]:
X['AGE'].agg(['min', 'max', 'mean', 'median', 'std']) 

In [ ]:
# Display Age outliers...

plt.figure(figsize=(17, 5))
sns.boxplot(data=X, x='AGE')
plt.title('Outliers and wrong values of Customers Ages')
plt.show()

In [ ]:
X = X.drop(['AGE'], axis=1)
X_test = X_test.drop(['AGE'], axis=1)

In [ ]:
import matplotlib.pyplot as plt

high_spent = X_test[X_test['SPEND']=='Low']
occupation_counts = high_spent['OCCUPATION'].value_counts().reset_index()
occupation_counts.columns = ['OCCUPATION', 'Count']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(occupation_counts['OCCUPATION'], occupation_counts['Count'], color=plt.cm.viridis(occupation_counts['Count']/occupation_counts['Count'].max()))
ax.set_xlabel('Number of Clients')
ax.set_ylabel('Occupation')
ax.set_title('Count of Occupation for Low Spent')
plt.colorbar(plt.cm.ScalarMappable(cmap='viridis'), ax=ax, label='Number of Clients')
plt.tight_layout()
plt.show()

In [ ]:
def repl_values(df):
    df['SPEND'] = df['SPEND'].replace({'High':1, 'Low':0})
    df['VALUE_PAYMENTS'] = df['VALUE_PAYMENTS'].replace({'Small':0, 'Medium':1, 'Large':2})

repl_values(X)
repl_values(X_test)

In [ ]:
X['OCCUPATION'].unique()

In [ ]:
X['OCCUPATION'] = X['OCCUPATION'].replace(['_______'], 'Others')
X_test['OCCUPATION'] = X_test['OCCUPATION'].replace(['_______'], 'Others')

In [ ]:
cols = ['NO_OF_LOAN', 'SPEND', 'VALUE_PAYMENTS', 'CREDIT_HISTORY_MONTHS', 
        'NUM_BANK_ACCOUNTS', 'NUM_CREDIT_CARD', 'INTEREST_RATE', 
        'DELAY_FROM_DUE_DATE', 'NUM_OF_DELAYED_PAYMENT', 'NUM_CREDIT_INQUIRIES']

def new_f(df, cols):
    cols_exist = [c for c in cols if c in df.columns]
    if cols_exist:
        df['NEW_COUNT_FEATURE'] = df[cols_exist].gt(0).sum(axis=1)

new_f(X, cols)
new_f(X_test, cols)
print("Done!")

In [ ]:
X.head()

Check target balance

In [ ]:
import matplotlib.pyplot as plt

if 'CREDIT_SCORE' in X.columns:
    target_data = X['CREDIT_SCORE']
elif isinstance(y, pd.DataFrame):
    target_data = y.iloc[:, 0]
else:
    target_data = y

counts = target_data.value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90)
ax.set_title('Percentages of target labels', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
import numpy as np

y1 = X['CREDIT_SCORE'].copy() if 'CREDIT_SCORE' in X.columns else y.copy()
y1 = y1.replace({'Poor': 0, 'Standard': 1, 'Good': 2})
X_train = X.drop(['CREDIT_SCORE'], axis=1, errors='ignore').copy()

cat_cols = X_train.select_dtypes('object').columns.tolist()
num_cols = X_train.select_dtypes(include=['float64', 'int64', 'Int64']).columns.tolist()

print(f"Cat cols: {cat_cols}")
print(f"Num cols: {num_cols}")
print(f"X_train shape: {X_train.shape}")
print(f"y1 shape: {y1.shape}")

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipe, cat_cols),
    ('num', num_pipe, num_cols)
], remainder='passthrough')

XGBClf = Pipeline([
    ('preprocess', preprocessor),
    ('xgb', XGBClassifier(learning_rate=0.01, random_state=42, n_jobs=-1))
])

xg_scores = cross_validate(estimator=XGBClf, X=X_train, y=y1, cv=5,
                           scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
                           return_train_score=True, n_jobs=-1)

xg_scores_df = pd.DataFrame(xg_scores)
xg_scores_df

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RFClf = Pipeline([
    ('preprocess', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, class_weight='balanced', n_jobs=-1))
])

random_scores = cross_validate(estimator=RFClf, X=X_train, y=y1, cv=5,
                               scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
                               return_train_score=True, n_jobs=-1)

random_scores_df = pd.DataFrame(random_scores)
random_scores_df

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

HGBClf = Pipeline([
    ('preprocess', preprocessor),
    ('hgb', HistGradientBoostingClassifier(max_iter=100, max_depth=10, random_state=42, class_weight='balanced'))
])

hist_scores = cross_validate(estimator=HGBClf, X=X_train, y=y1, cv=5,
                             scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
                             return_train_score=True, n_jobs=-1)

hist_scores_df = pd.DataFrame(hist_scores)
hist_scores_df

In [ ]:
Report = pd.DataFrame()

all_models = {'XGBoost': xg_scores_df, 'Random Forest': random_scores_df, 'Hist Gradient Boosting': hist_scores_df}
for model, scores in all_models.items():
    metrics = {
        'Train Mean Accuracy': scores['train_accuracy'].mean(),
        'Test Mean Accuracy': scores['test_accuracy'].mean(),
        'Train Mean Precision': scores['train_precision_macro'].mean(),
        'Test Mean Precision': scores['test_precision_macro'].mean(),
        'Train Mean Recall': scores['train_recall_macro'].mean(),
        'Test Mean Recall': scores['test_recall_macro'].mean(),
        'Train Mean F1': scores['train_f1_macro'].mean(),
        'Test Mean F1': scores['test_f1_macro'].mean(),
    }
    Report[model] = metrics

Report = Report.round(4)
Report = Report.reset_index().rename(columns={'index': 'Metric'})
Report

Model Evolution

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

X_tr, X_valid, y_tr, y_valid = train_test_split(X_train, y1, test_size=0.3, random_state=42)

best_model = HGBClf
best_model.fit(X_tr, y_tr)

y_pred = best_model.predict(X_valid)
y_pred_proba = best_model.predict_proba(X_valid)

classes = best_model.classes_
y_valid_bin = label_binarize(y_valid, classes=classes)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(confusion_matrix(y_valid, y_pred), annot=True, fmt='d', ax=ax[0], xticklabels=classes, yticklabels=classes)
ax[0].tick_params(axis='x', rotation=45)
ax[0].tick_params(axis='y', rotation=0)
ax[0].set_title('Confusion matrix of HGB prediction')
ax[0].set_xlabel('Predicted')
ax[0].set_ylabel('Actual')

colors = ['blue', 'orange', 'green']
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_valid_bin[:, i], y_pred_proba[:, i])
    auc_score = roc_auc_score(y_valid_bin[:, i], y_pred_proba[:, i])
    ax[1].plot(fpr, tpr, label=f'{cls} AUC = {auc_score:.3f}', color=colors[i], lw=2)

ax[1].plot([0,1], [0,1], linestyle='--', color='gray')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curve')
ax[1].legend()
ax[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Fazer previsões no conjunto de teste
prediction = best_model.predict(X_test)

# Mapear valores numéricos de volta para labels
label_map = {0: 'Poor', 1: 'Standard', 2: 'Good'}
prediction_labels = [label_map.get(p, p) for p in prediction]

# Criar DataFrame com resultado
# submission = pd.DataFrame({'ID':test['ID'].reset_index(drop=True), 'Credit_Score':prediction})
# submission.to_csv('Credit_Score_Prediction.csv', index=False)
submission = X_test.copy()
submission['CREDIT_SCORE_PREDICTED'] = prediction_labels

print('Prediction Classes Counts:')
print(pd.Series(prediction_labels).value_counts(), '\n')

# Salvar tabela no Snowflake
submission_snowpark = session.create_dataframe(submission)
submission_snowpark.write.mode('overwrite').save_as_table('ACCEPTION.CREDIT_SCORE.CREDIT_SCORE_FINAL')

print('Tabela CREDIT_SCORE_FINAL criada com sucesso!')
print(f'Total de registros: {submission_snowpark.count()}')

submission.head(10)